# MoneyAI 💰

Agente de contabilidad personal que analiza tus movimientos bancarios desde Gmail.


In [56]:
import os.path
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# Scopes para Gmail y Google Sheets
SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/spreadsheets'
]

def get_google_credentials():
    """Obtiene las credenciales de Gmail, solicitando autorización si es necesario."""
    creds = None
    
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    return creds

# Autenticar con Google (Gmail + Sheets)
creds = get_google_credentials()

# Servicios de Google
gmail_service = build('gmail', 'v1', credentials=creds)
sheets_service = build('sheets', 'v4', credentials=creds)

# ID del Google Sheet
SPREADSHEET_ID = "1q5Br4HnBl8PyugdNJuCuIFTWg3gIPas-Yoxq4yQPOcY"

print("✅ Conectado a Gmail y Google Sheets correctamente")

✅ Conectado a Gmail y Google Sheets correctamente


In [57]:
import base64
import re

def extraer_monto(texto):
    """Extrae el monto de un texto usando regex."""
    # Patrones para diferentes formatos de montos
    patrones = [
        r'Monto\s*\$?\s*([\d.,]+)',  # Monto $1.234,56 o Monto 1234
        r'Monto\s+U\$S\s*([\d.,]+)',  # Monto U$S 14,16
        r'\$\s*([\d.,]+)',  # $1.234,56
        r'U\$S\s*([\d.,]+)',  # U$S 14,16
        r'ARS\s*([\d.,]+)',  # ARS 1234
        r'USD\s*([\d.,]+)',  # USD 1234
    ]
    
    for patron in patrones:
        match = re.search(patron, texto, re.IGNORECASE)
        if match:
            monto = match.group(1)
            # Determinar moneda
            if 'U$S' in texto.upper() or 'USD' in texto.upper():
                return f"U$S {monto}"
            else:
                return f"${monto}"
    return None

def limpiar_html(html_text):
    """Limpia tags HTML y devuelve texto plano."""
    texto = re.sub(r'<[^>]+>', ' ', html_text)
    texto = texto.replace('&nbsp;', ' ').replace('&amp;', '&')
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def extraer_cuerpo_email(payload):
    """Extrae el cuerpo de texto del email recursivamente."""
    texto_plano = ""
    html = ""
    
    if 'body' in payload and payload['body'].get('data'):
        data = payload['body']['data']
        contenido = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        mime_type = payload.get('mimeType', '')
        if 'html' in mime_type:
            html = contenido
        else:
            texto_plano = contenido
    
    if 'parts' in payload:
        for part in payload['parts']:
            mime_type = part.get('mimeType', '')
            if part['body'].get('data'):
                data = part['body']['data']
                contenido = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
                if mime_type == 'text/plain':
                    texto_plano = contenido
                elif mime_type == 'text/html':
                    html = contenido
            elif mime_type.startswith('multipart/'):
                resultado = extraer_cuerpo_email(part)
                if resultado:
                    return resultado
    
    if texto_plano.strip():
        return texto_plano
    elif html:
        return limpiar_html(html)
    return ""

def buscar_emails(query, max_results=50):
    """Busca emails en Gmail según una query y devuelve lista de emails procesados."""
    results = gmail_service.users().messages().list(userId='me', q=query, maxResults=max_results).execute()
    messages = results.get('messages', [])
    
    emails = []
    for msg in messages:
        message = gmail_service.users().messages().get(userId='me', id=msg['id'], format='full').execute()
        headers = message['payload'].get('headers', [])
        
        # Extraer contenido y monto
        contenido = extraer_cuerpo_email(message['payload'])
        snippet = message.get('snippet', '')
        
        # Buscar monto en contenido o snippet
        monto = extraer_monto(contenido) or extraer_monto(snippet)
        
        emails.append({
            'id': msg['id'],
            'subject': next((h['value'] for h in headers if h['name'] == 'Subject'), 'Sin asunto'),
            'from': next((h['value'] for h in headers if h['name'] == 'From'), 'Desconocido'),
            'date': next((h['value'] for h in headers if h['name'] == 'Date'), 'Sin fecha'),
            'snippet': snippet,
            'contenido': contenido[:2000] if contenido else snippet,  # Más contenido para el LLM
            'monto_extraido': monto,  # ✅ Monto extraído directamente
            'payload': message['payload']
        })
    
    return emails

def mostrar_emails(emails):
    """Muestra los emails de forma legible."""
    print(f"📧 Se encontraron {len(emails)} emails\n")
    for i, email in enumerate(emails):
        print(f"--- Email {i+1} ---")
        print(f"📌 Asunto: {email['subject']}")
        print(f"👤 De: {email['from']}")
        print(f"📅 Fecha: {email['date']}")
        print(f"📝 Resumen: {email['snippet'][:150]}...")
        print()


In [58]:
# Buscar avisos de Santander
santander_emails = buscar_emails('from:mensajesyavisos@mails.santander.com.ar')
mostrar_emails(santander_emails)


📧 Se encontraron 50 emails

--- Email 1 ---
📌 Asunto: Se realizó un débito en tu cuenta
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Sat, 3 Jan 2026 11:21:41 -0300 (ART)
📝 Resumen: Información sobre el débito en tu cuenta por recurrencia Hola GARCIA BARRIOLA LEANDRO OMAR NICOLAS, Queremos comunicarte que se realizó un débito en t...

--- Email 2 ---
📌 Asunto: Aviso de transferencia
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Fri, 2 Jan 2026 17:05:39 -0300 (ART)
📝 Resumen: Información sobre tu transferencia Se realizó la siguiente transferencia a tu nombre: Destinatario 20348750454 Cuenta de origen Cuenta en Pesos XXX-XX...

--- Email 3 ---
📌 Asunto: Aviso de transferencia
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Fri, 2 Jan 2026 10:17:05 -0300 (ART)
📝 Resumen: Información sobre tu transferencia Se realizó la siguiente transferencia a tu nombre: Destinatario 20922973424 Cuenta de origen Cuenta en Pesos 

In [59]:
# TEST: Verificar conexión con Google Sheets API
import requests

print("🔍 Test de conexión a Google Sheets...")

# Test 1: Conexión directa
try:
    r = requests.get("https://sheets.googleapis.com", timeout=10)
    print(f"✅ Conexión HTTP: {r.status_code}")
except Exception as e:
    print(f"❌ Error HTTP: {e}")

# Test 2: Verificar credenciales
print(f"\n📋 Credenciales válidas: {creds.valid}")
print(f"📋 Credenciales expiradas: {creds.expired if hasattr(creds, 'expired') else 'N/A'}")
print(f"📋 Scopes: {creds.scopes}")


🔍 Test de conexión a Google Sheets...
✅ Conexión HTTP: 404

📋 Credenciales válidas: True
📋 Credenciales expiradas: False
📋 Scopes: ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/spreadsheets']


In [60]:
import os
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que existe la API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ Falta OPENAI_API_KEY en el archivo .env")

# Inicializar el modelo (gpt-4o-mini es rápido y económico)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Modelo OpenAI (gpt-4o-mini) configurado correctamente")


✅ Modelo OpenAI (gpt-4o-mini) configurado correctamente


In [61]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

# Definir las categorías
class CategoriaEmail(str, Enum):
    TRANSFERENCIA_MP_COMIDAS = "transferencia_mp_comidas"
    FONDEO_COCOS_CAPITAL = "fondeo_cocos_capital"
    DEBITO_CUENTA = "debito_en_cuenta"
    DEBITO_TARJETA_CREDITO = "debito_pago_tarjeta_credito"
    DEBITO_PRESTAMO = "debito_automatico_prestamo"
    BENEFICENCIA = "beneficencia"
    OTRO = "otro"

# Schema para la respuesta estructurada
class EmailCategorizado(BaseModel):
    categoria: CategoriaEmail = Field(description="Categoría del email bancario")
    monto: Optional[str] = Field(default=None, description="Monto de la transacción si está disponible")
    descripcion: str = Field(description="Breve descripción de la transacción")

# Crear el modelo con salida estructurada
llm_categorizer = llm.with_structured_output(EmailCategorizado)

def categorizar_email(email: dict) -> dict:
    """Categoriza un email bancario usando el LLM."""
    
    prompt = f"""Analiza este email bancario de Santander y categorízalo.

CATEGORÍAS DISPONIBLES:
- transferencia_mp_comidas: Transferencias a MercadoPago para comidas. CRITERIOS: Servicio = "FONDEO" Y cuenta destino contiene "0077" (ARS XXX-XXX0077)
- fondeo_cocos_capital: Transferencias a Fondeo Cocos Capital. CRITERIOS: CBU de Destino = "0000053600000028204257"
- debito_en_cuenta: Débitos directos en cuenta (no tarjeta, no préstamo)
- debito_pago_tarjeta_credito: Pagos con tarjeta de crédito (AMEX o Visa)
- debito_automatico_prestamo: Débitos automáticos de cuotas de préstamos
- beneficencia: Transferencias a MercadoPago para ayudar. CRITERIOS: Destinatario = "20922973424"
- otro: Cualquier otro tipo (promociones, avisos informativos, transferencias que NO cumplan los criterios anteriores, etc.)

IMPORTANTE: Para "transferencia_mp_comidas" AMBOS criterios deben cumplirse (FONDEO + cuenta 0077).

EMAIL A ANALIZAR:
Asunto: {email['subject']}
Contenido: {email.get('contenido', email['snippet'])}

Categoriza este email y extrae el monto si está disponible."""

    resultado = llm_categorizer.invoke(prompt)
    
    return {
        **email,
        'categoria': resultado.categoria.value,
        'monto': email.get('monto_extraido'),  # Usar el monto extraído por regex
        'descripcion_ia': resultado.descripcion
    }

print("✅ Función de categorización lista")


✅ Función de categorización lista


In [62]:
# Definir el estado del agente
class AgentState(TypedDict):
    emails_pendientes: List[dict]
    emails_categorizados: List[dict]
    email_actual: Optional[dict]
    indice: int

# Nodos del grafo
def inicializar(state: AgentState) -> AgentState:
    """Inicializa el procesamiento."""
    return {
        **state,
        "indice": 0,
        "emails_categorizados": []
    }

def seleccionar_email(state: AgentState) -> AgentState:
    """Selecciona el siguiente email a procesar."""
    idx = state["indice"]
    if idx < len(state["emails_pendientes"]):
        return {
            **state,
            "email_actual": state["emails_pendientes"][idx]
        }
    return {**state, "email_actual": None}

def procesar_email(state: AgentState) -> AgentState:
    """Procesa y categoriza el email actual."""
    email = state["email_actual"]
    if email:
        print(f"  📧 Procesando: {email['subject'][:50]}...")
        email_categorizado = categorizar_email(email)
        return {
            **state,
            "emails_categorizados": state["emails_categorizados"] + [email_categorizado],
            "indice": state["indice"] + 1
        }
    return state

def hay_mas_emails(state: AgentState) -> str:
    """Determina si hay más emails por procesar."""
    if state["indice"] < len(state["emails_pendientes"]):
        return "seleccionar"
    return END

# Construir el grafo
workflow = StateGraph(AgentState)

# Agregar nodos
workflow.add_node("inicializar", inicializar)
workflow.add_node("seleccionar", seleccionar_email)
workflow.add_node("procesar", procesar_email)

# Definir flujo
workflow.set_entry_point("inicializar")
workflow.add_edge("inicializar", "seleccionar")
workflow.add_edge("seleccionar", "procesar")
workflow.add_conditional_edges("procesar", hay_mas_emails)

# Compilar el agente
agente_categorizador = workflow.compile()

print("✅ Agente de categorización compilado")


✅ Agente de categorización compilado


In [63]:
# Ejecutar el agente con los emails de Santander
import time

# Cantidad de emails a procesar (reducir si hay errores de cuota)
CANTIDAD_EMAILS = 20

print(f"🚀 Iniciando categorización de {CANTIDAD_EMAILS} emails...\n")

resultado = agente_categorizador.invoke(
    {
        "emails_pendientes": santander_emails[:CANTIDAD_EMAILS],
        "emails_categorizados": [],
        "email_actual": None,
        "indice": 0
    },
    config={"recursion_limit": 150}  # Aumentar límite para procesar más emails
)

print("\n✅ Categorización completada!")
print(f"📊 Total procesados: {len(resultado['emails_categorizados'])}")


🚀 Iniciando categorización de 20 emails...

  📧 Procesando: Se realizó un débito en tu cuenta...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de alta de destinatario de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de débito automático...
  📧 Procesando: ¿Compraste y no ahorraste? ¡Hacelo ahora con Super...
  📧 Procesando: Pagaste $7.000,00...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Pagaste $7.863,79...
  📧 Procesando: Se realizó un débito en tu cuenta...
  📧 Procesando: Aviso de débito automático...
  📧 Procesando: Pagaste $14.700,00...

✅ Categorización completada!
📊 Total procesados

In [64]:
# Mostrar resultados agrupados por categoría
from collections import defaultdict

emails_por_categoria = defaultdict(list)
for email in resultado['emails_categorizados']:
    emails_por_categoria[email['categoria']].append(email)

# Emojis por categoría
emojis = {
    'transferencia_mp_comidas': '🍔',
    'debito_en_cuenta': '🏦',
    'debito_pago_tarjeta_credito': '💳',
    'debito_automatico_prestamo': '📋',
    'beneficencia': '👨‍👩‍👧‍👦',
    'otro': '📌'
}

# Mostrar resumen
print("=" * 60)
print("📊 RESUMEN DE EMAILS CATEGORIZADOS")
print("=" * 60)

for categoria, emails in emails_por_categoria.items():
    emoji = emojis.get(categoria, '📧')
    print(f"\n{emoji} {categoria.upper().replace('_', ' ')} ({len(emails)} emails)")
    print("-" * 40)
    for email in emails:
        monto = f" - {email['monto']}" if email.get('monto') else ""
        print(f"  • {email['subject'][:45]}...{monto}")
        print(f"    └─ {email['descripcion_ia'][:60]}...")

print("\n" + "=" * 60)


📊 RESUMEN DE EMAILS CATEGORIZADOS

🍔 TRANSFERENCIA MP COMIDAS (2 emails)
----------------------------------------
  • Se realizó un débito en tu cuenta... - $150000.0
    └─ Débito en cuenta asociado a la recurrencia con MERCADOLIBRE ...
  • Se realizó un débito en tu cuenta... - $50000.0
    └─ Débito en cuenta asociado a la recurrencia con MERCADOLIBRE ...

📧 FONDEO COCOS CAPITAL (1 emails)
----------------------------------------
  • Aviso de transferencia... - $1.500.000,00
    └─ Transferencia a CBU de Destino 0000053600000028204257...

👨‍👩‍👧‍👦 BENEFICENCIA (1 emails)
----------------------------------------
  • Aviso de transferencia... - $330.000,00
    └─ Transferencia a MercadoPago para ayudar....

📌 OTRO (11 emails)
----------------------------------------
  • Aviso de alta de destinatario de transferenci...
    └─ Aviso de alta de nuevo destinatario de transferencias....
  • Aviso de transferencia... - $40.000,00
    └─ Transferencia a destinatario 27217636537 desde cuenta X

In [65]:
# Leer estructura del Google Sheet
import time

# Reintentar conexión
for intento in range(3):
    try:
        print(f"Intentando conectar... (intento {intento + 1})")
        spreadsheet = sheets_service.spreadsheets().get(spreadsheetId=SPREADSHEET_ID).execute()
        break
    except Exception as e:
        print(f"Error: {e}")
        if intento < 2:
            time.sleep(2)
        else:
            raise

print("\n📊 ESTRUCTURA DEL GOOGLE SHEET")
print("=" * 60)
print(f"Título: {spreadsheet['properties']['title']}")
print(f"\n📑 Hojas disponibles:")

for sheet in spreadsheet['sheets']:
    title = sheet['properties']['title']
    sheet_id = sheet['properties']['sheetId']
    print(f"  • {title} (ID: {sheet_id})")

# Leer headers de la primera hoja
primera_hoja = spreadsheet['sheets'][0]['properties']['title']
headers = sheets_service.spreadsheets().values().get(
    spreadsheetId=SPREADSHEET_ID,
    range=f"'{primera_hoja}'!A1:Z1"
).execute()

print(f"\n📋 Columnas de '{primera_hoja}':")
if headers.get('values'):
    for i, col in enumerate(headers['values'][0], 1):
        print(f"  {i}. {col}")


Intentando conectar... (intento 1)

📊 ESTRUCTURA DEL GOOGLE SHEET
Título: proyecto personal_dev

📑 Hojas disponibles:
  • estructura financiera (ID: 971113873)
  • Económico (ID: 1485142438)
  • gastos (ID: 2119808242)
  • proyecciones economicas (ID: 1280362112)
  • Inversión en educación (ID: 370654592)
  • deuda facultad (ID: 37067304)
  • Presup Educación (ID: 322406841)
  • spread auto (ID: 922431744)
  • Préstamo tasa fija (ID: 1667097035)
  • prestamo UVA (ID: 387252814)

📋 Columnas de 'estructura financiera':
  1. 
  2. 
  3. 
  4. 
  5. 
  6. 
  7. 
  8. 
  9. 2021
  10. 2021
  11. 2021
  12. 2021
  13. 2021
  14. 2021
  15. 2021
  16. 2021
  17. 2021
  18. 2021
  19. 2021
  20. 2021
  21. 2022
  22. 2022
  23. 2022
  24. 2022
  25. 2022
  26. 2022


In [68]:
# Crear hoja "transacciones" como base de datos
import re
from datetime import datetime
from email.utils import parsedate_to_datetime

HOJA_TRANSACCIONES = "transacciones"

def parsear_monto(monto_str):
    """Convierte string de monto a número."""
    if not monto_str:
        return 0
    monto = re.sub(r'[^\d.,]', '', str(monto_str))
    if ',' in monto and '.' in monto:
        if monto.rfind(',') > monto.rfind('.'):
            monto = monto.replace('.', '').replace(',', '.')
        else:
            monto = monto.replace(',', '')
    elif ',' in monto:
        monto = monto.replace(',', '.')
    try:
        return float(monto)
    except:
        return 0

def parsear_fecha(fecha_str):
    """Convierte fecha de email a formato YYYY-MM-DD."""
    try:
        dt = parsedate_to_datetime(fecha_str)
        return dt.strftime("%Y-%m-%d")
    except:
        return fecha_str[:10] if fecha_str else ""

def crear_hoja_transacciones():
    """Crea la hoja 'transacciones' si no existe."""
    # Verificar si ya existe
    spreadsheet = sheets_service.spreadsheets().get(spreadsheetId=SPREADSHEET_ID).execute()
    hojas_existentes = [s['properties']['title'] for s in spreadsheet['sheets']]
    
    if HOJA_TRANSACCIONES in hojas_existentes:
        print(f"✅ Hoja '{HOJA_TRANSACCIONES}' ya existe")
        return True
    
    # Crear la hoja
    sheets_service.spreadsheets().batchUpdate(
        spreadsheetId=SPREADSHEET_ID,
        body={
            "requests": [{
                "addSheet": {
                    "properties": {"title": HOJA_TRANSACCIONES}
                }
            }]
        }
    ).execute()
    
    # Agregar headers
    headers = [["Fecha", "Categoría", "Monto", "Descripción", "Email ID", "Asunto"]]
    sheets_service.spreadsheets().values().update(
        spreadsheetId=SPREADSHEET_ID,
        range=f"'{HOJA_TRANSACCIONES}'!A1:F1",
        valueInputOption="RAW",
        body={"values": headers}
    ).execute()
    
    print(f"✅ Hoja '{HOJA_TRANSACCIONES}' creada con headers")
    return True

def agregar_transacciones(emails_categorizados):
    """Agrega los emails categorizados como filas en la hoja transacciones (sin duplicados)."""
    
    # Obtener IDs de emails ya registrados para evitar duplicados
    result = sheets_service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"'{HOJA_TRANSACCIONES}'!E:E"  # Columna de Email ID
    ).execute()
    ids_existentes = set()
    for row in result.get('values', [])[1:]:  # Skip header
        if row:
            ids_existentes.add(row[0])
    
    print(f"📋 Ya hay {len(ids_existentes)} transacciones registradas")
    
    # Preparar filas (solo las nuevas)
    filas = []
    duplicados = 0
    for email in emails_categorizados:
        if email.get('categoria') != 'otro':  # Ignorar "otro"
            email_id = email.get('id', '')
            if email_id in ids_existentes:
                duplicados += 1
                continue  # Skip duplicado
            
            fila = [
                parsear_fecha(email.get('date', '')),
                email.get('categoria', ''),
                parsear_monto(email.get('monto')),
                email.get('descripcion_ia', '')[:100],
                email_id,
                email.get('subject', '')[:50]
            ]
            filas.append(fila)
    
    if duplicados > 0:
        print(f"⏭️ Saltando {duplicados} emails ya registrados")
    
    if not filas:
        print("✅ No hay transacciones nuevas para agregar")
        return 0
    
    # Encontrar la última fila con datos
    result = sheets_service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=f"'{HOJA_TRANSACCIONES}'!A:A"
    ).execute()
    ultima_fila = len(result.get('values', [])) + 1
    
    # Agregar filas nuevas
    sheets_service.spreadsheets().values().update(
        spreadsheetId=SPREADSHEET_ID,
        range=f"'{HOJA_TRANSACCIONES}'!A{ultima_fila}:F{ultima_fila + len(filas) - 1}",
        valueInputOption="USER_ENTERED",
        body={"values": filas}
    ).execute()
    
    return len(filas)

print("📊 Sistema de base de datos en Google Sheets")
print("=" * 60)


📊 Sistema de base de datos en Google Sheets


In [69]:
# Crear hoja y agregar transacciones
print("🚀 Creando base de datos de transacciones...")
print("=" * 60)

# 1. Crear la hoja si no existe
crear_hoja_transacciones()

# 2. Agregar los emails categorizados
print(f"\n📝 Agregando {len(resultado['emails_categorizados'])} emails categorizados...")
cantidad = agregar_transacciones(resultado['emails_categorizados'])

print(f"\n✅ ¡Listo! Se agregaron {cantidad} transacciones a la hoja '{HOJA_TRANSACCIONES}'")
print(f"\n📊 Ahora podés usar fórmulas en tu sheet 'gastos' para traer los datos.")
print(f"\nEjemplo de fórmula para sumar 'beneficencia' de Enero 2026:")
print(f'=SUMIFS(transacciones!C:C, transacciones!B:B, "beneficencia", transacciones!A:A, ">=2026-01-01", transacciones!A:A, "<=2026-01-31")')


🚀 Creando base de datos de transacciones...
✅ Hoja 'transacciones' ya existe

📝 Agregando 20 emails categorizados...
📋 Ya hay 9 transacciones registradas
⏭️ Saltando 9 emails ya registrados
✅ No hay transacciones nuevas para agregar

✅ ¡Listo! Se agregaron 0 transacciones a la hoja 'transacciones'

📊 Ahora podés usar fórmulas en tu sheet 'gastos' para traer los datos.

Ejemplo de fórmula para sumar 'beneficencia' de Enero 2026:
=SUMIFS(transacciones!C:C, transacciones!B:B, "beneficencia", transacciones!A:A, ">=2026-01-01", transacciones!A:A, "<=2026-01-31")
